# EOIR Decisions Scraper

Scrapes two categories of immigration law decisions from the DOJ Executive Office for Immigration Review:

- **AG/BIA Precedent Decisions** — Board of Immigration Appeals and Attorney General rulings, Volumes 24–29 (~540 decisions)
- **OCAHO Published Decisions** — Office of the Chief Administrative Hearing Officer, Looseleaf Volumes 14–22 (~1,200 decision rows)

**Source:** https://www.justice.gov/eoir/ag-bia-decisions

**Output files:**
- `data/bia_ag_decisions.json` — BIA/AG decisions, ~22 fields per row
- `data/ocaho_decisions.json` — OCAHO decisions, ~22 fields per row
- `data/changelogs/YYYY-MM-DD_bia_ag.json` — daily change log for BIA/AG
- `data/changelogs/YYYY-MM-DD_ocaho.json` — daily change log for OCAHO
- `data/error_logs/YYYY-MM-DD_*.json` — per-volume error logs

**Scheduling:** Run daily with:
```
# Add to crontab (crontab -e):
# 0 7 * * * cd /path/to/ImmigrationScraper && /opt/homebrew/opt/python@3.12/bin/jupyter nbconvert --to notebook --execute eoir_decisions_scraper.ipynb >> data/cron.log 2>&1
```

**Kernel:** Use Python 3.12 (`/opt/homebrew/opt/python@3.12/bin/python3.12`)

In [ ]:
###############################################################
# CELL 1: EXTRACTION FUNCTIONS
# Handles parsing of both BIA/AG volume pages and OCAHO listing
# pages. All scraping logic is hyper-specific to DOJ EOIR HTML.
###############################################################

import re
import os
import shutil
import hashlib
import subprocess
import tempfile
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.justice.gov"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Locate pdftotext — works on macOS (Homebrew) and Linux (apt poppler-utils)
PDFTOTEXT = (
    shutil.which('pdftotext') or       # in PATH (Linux CI, or mac after brew link)
    '/opt/homebrew/bin/pdftotext'      # macOS Homebrew fallback
)


def make_absolute_url(href):
    """Normalize any href (relative or absolute) to a full justice.gov URL."""
    if not href:
        return None
    href = href.strip()
    if href.startswith('http'):
        return href
    if href.startswith('/'):
        return BASE_URL + href
    return BASE_URL + '/' + href


def extract_pdf_text(pdf_url):
    """
    Download a PDF and extract its full text using pdftotext (poppler).
    Returns (text_string, success_bool).
    Install: brew install poppler  OR  sudo apt-get install poppler-utils
    """
    try:
        resp = requests.get(pdf_url, headers=HEADERS, timeout=45, stream=True)
        resp.raise_for_status()
        with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp:
            for chunk in resp.iter_content(chunk_size=8192):
                tmp.write(chunk)
            tmp_path = tmp.name
        result = subprocess.run(
            [PDFTOTEXT, '-layout', tmp_path, '-'],
            capture_output=True, text=True, timeout=60
        )
        os.unlink(tmp_path)
        if result.returncode == 0:
            return result.stdout.strip(), True
        return "", False
    except Exception:
        return "", False


def extract_bia_decisions(page_html, volume_name, volume_url, volume_num, download_pdfs=False):
    """
    Parse a BIA/AG precedent-decisions volume page and return a list of decision dicts.

    Handles two table formats found across volumes:
      - class='table'         (Volume 29, newest)
      - class='no-background' (Volumes 24-28, legacy redirect pages)

    Each decision on these pages is structured as:
      <table>                   <- one table per decision
        <tr>
          <td><strong>CASE_NAME</strong>, VOLUME I&N Dec. PAGE (BODY YEAR)</td>
          <td><a href="PDF_LINK">ID NNNN</a> (PDF)</td>
        </tr>
      </table>
      <p>Headnote / holding summary paragraph(s)...</p>
      <hr>  <- separator before next decision

    Deciding body values seen in practice: BIA, A.G, A.G., AAO, DIR, BIA Dec
    """
    soup = BeautifulSoup(page_html, "html.parser")
    decisions = []

    # Find main content body — modern pages use field_body, legacy use bodytext
    body_div = (
        soup.find(class_="field_body") or
        soup.find(class_="bodytext") or
        soup.find(class_=re.compile(r"field.body"))
    )
    if not body_div:
        return decisions

    all_tables = body_div.find_all('table')

    for table in all_tables:
        try:
            rows = table.find_all('tr')
            if not rows:
                continue
            row = rows[0]
            cells = row.find_all('td')
            if len(cells) < 2:
                continue

            first_cell = cells[0]
            second_cell = cells[-1]  # last cell holds the PDF link

            # Case name comes from <strong> or <b> tag in first cell
            bold_tag = first_cell.find(['strong', 'b'])
            if not bold_tag:
                continue
            case_name = bold_tag.get_text(strip=True).rstrip(',').rstrip(';').rstrip('.').strip()
            if not case_name or len(case_name) < 2:
                continue

            raw_cell_text = first_cell.get_text(" ", strip=True)

            # Parse citation: "VOLUME I&N Dec[.] PAGE (BODY[,] YEAR)"
            # Examples: "29 I&N Dec 680 (BIA 2026)"
            #           "26 I&N Dec. 922 (BIA 2017)"
            #           "28 I&N Dec. 883 (BIA 2025)"
            #           "27 I&N Dec. 271 (A.G. 2018)"  <- AG with trailing dot
            #           "28 I&N Dec. 7 (A.G 2020)"     <- AG without trailing dot
            citation_m = re.search(
                r'(\d+)\s+I[&N]+\s+Dec\.?\s+(\d+)\s+\(([^,)]+?)[,.]?\s*(\d{4})\)',
                raw_cell_text
            )
            if citation_m:
                cit_volume  = int(citation_m.group(1))
                cit_page    = int(citation_m.group(2))
                cit_body    = citation_m.group(3).strip()
                cit_year    = int(citation_m.group(4))
                citation_full = citation_m.group(0)
            else:
                cit_volume    = volume_num
                cit_page      = None
                cit_body      = "BIA"
                cit_year      = None
                citation_full = raw_cell_text[:120].strip()

            # Deciding body classification
            # Note: A.G matches both "A.G." (with trailing dot) and "A.G" (without)
            is_attorney_general = bool(re.search(
                r'A\.G\.?|Att\'y\s+Gen|Attorney\s+General', cit_body, re.I))
            is_aao = bool(re.search(r'\bAAO\b|A\.A\.O\.', cit_body, re.I))
            is_director = bool(re.search(r'\bDIR\b', cit_body, re.I))
            deciding_body_clean = (
                "Attorney General" if is_attorney_general else
                "AAO"              if is_aao else
                "Director"         if is_director else
                "BIA"
            )

            # PDF link from second cell
            pdf_link = second_cell.find('a')
            if not pdf_link:
                continue
            raw_href = pdf_link.get('href', '')
            pdf_url  = make_absolute_url(raw_href)

            # Decision ID (numeric) from link text: "ID 4203" or "ID 3886"
            id_m = re.search(r'ID\s*#?\s*(\d+)', second_cell.get_text())
            decision_id = int(id_m.group(1)) if id_m else None

            # Media/file ID from the PDF URL path
            media_id_m = (
                re.search(r'/media/(\d+)/', raw_href) or      # /media/XXXXXX/dl format
                re.search(r'/d9/\d{4}-\d{2}/(.+)\.pdf', raw_href) or  # /d9/YYYY-MM/NNNN.pdf
                re.search(r'/(\d{4,})\.pdf', raw_href)        # /vol25/3765.pdf
            )
            pdf_media_id = media_id_m.group(1) if media_id_m else None

            # Headnote: one or more <p> tags immediately after this table, until <hr> or next <table>
            headnote_parts = []
            sibling = table.next_sibling
            while sibling:
                if hasattr(sibling, 'name'):
                    if sibling.name == 'p':
                        text = sibling.get_text(" ", strip=True)
                        # Skip non-breaking spaces only
                        if text and text.replace('\xa0', '').strip():
                            headnote_parts.append(text.strip())
                    elif sibling.name in ('hr', 'table'):
                        break
                    elif sibling.name == 'div':
                        # Some legacy pages wrap each entry in a <div class="headline">
                        inner_p = sibling.find('p')
                        if inner_p:
                            text = inner_p.get_text(" ", strip=True)
                            if text and text.replace('\xa0', '').strip():
                                headnote_parts.append(text.strip())
                        break
                sibling = sibling.next_sibling
            headnote = " | ".join(headnote_parts)

            # Unique key is the PDF URL
            url = pdf_url or f"{BASE_URL}/eoir/bia-decision-id-{decision_id}"

            # Change-detection hash: case name + citation + headnote excerpt
            hash_source = f"{case_name}{citation_full}{headnote[:300]}"
            content_hash = hashlib.md5(hash_source.lower().encode('utf-8')).hexdigest()

            # Optional full PDF text
            full_text = ""
            pdf_download_attempted = False
            pdf_text_extracted = False
            text_word_count = 0

            if download_pdfs and pdf_url:
                pdf_download_attempted = True
                full_text, pdf_text_extracted = extract_pdf_text(pdf_url)
                text_word_count = len(full_text.split()) if full_text else 0

            decisions.append({
                "url":                    url,
                "content_hash":           content_hash,
                "decision_id":            decision_id,
                "case_name":              case_name,
                "citation_full":          citation_full,
                "citation_volume":        cit_volume,
                "citation_page":          cit_page,
                "citation_body":          cit_body,
                "citation_year":          cit_year,
                "deciding_body":          deciding_body_clean,
                "is_attorney_general":    is_attorney_general,
                "is_aao":                 is_aao,
                "is_director":            is_director,
                "pdf_url":                pdf_url,
                "pdf_media_id":           pdf_media_id,
                "headnote":               headnote,
                "headnote_word_count":    len(headnote.split()) if headnote else 0,
                "full_text":              full_text,
                "text_word_count":        text_word_count,
                "source_volume_name":     volume_name,
                "source_volume_url":      volume_url,
                "source_volume_number":   volume_num,
                "decision_type":          "BIA_AG",
                "raw_cell_text":          raw_cell_text,
                "pdf_download_attempted": pdf_download_attempted,
                "pdf_text_extracted":     pdf_text_extracted,
                "scraped_date":           TODAY_STR,
            })

        except Exception:
            continue  # Malformed table; outer loop handles volume-level errors

    return decisions


def extract_ocaho_decisions(page_html, volume_name, volume_url, volume_num, download_pdfs=False):
    """
    Parse an OCAHO volume listing page and return a list of decision dicts.

    Table structure (5 columns): CASE NAME | CASE NUMBER | DECISION DATE | REFERENCE NO. | CAHO DECISION

    Cases with multiple decisions use HTML rowspan on case name/number cells.
    BeautifulSoup only returns cells physically present in each <tr>, so we
    track current_case_name / current_case_number across rows:
      - 5 cells in row → new case (all fields present)
      - 3 cells in row → sub-decision (date + reference + caho; case info carried forward)
      - 4 cells in row → rare edge case (case_number rowspan differs from case_name rowspan)

    Case number format: YYYY[A|B|C]NNNNN
      A = Employer Sanctions (I-9 violations)
      B = Employee Discrimination (INA § 274B)
      C = Document Fraud (INA § 274C)
    """
    soup = BeautifulSoup(page_html, "html.parser")
    decisions = []

    body_div = (
        soup.find(class_="field_body") or
        soup.find(class_="bodytext")
    )
    if not body_div:
        return decisions

    table = body_div.find('table')
    if not table:
        return decisions

    current_case_name   = None
    current_case_number = None

    CASE_TYPE_MAP = {
        'A': 'Employer Sanctions (I-9 Violations)',
        'B': 'Employee Discrimination (INA § 274B)',
        'C': 'Document Fraud (INA § 274C)',
    }

    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if not cells:
            continue  # header row (<th>)

        # Drop empty/whitespace-only rows
        non_empty = [
            c for c in cells
            if c.get_text(strip=True).replace('\xa0', '').strip()
        ]
        if not non_empty:
            continue

        if len(cells) >= 5:
            current_case_name   = cells[0].get_text(" ", strip=True)
            current_case_number = cells[1].get_text(strip=True)
            date_cell  = cells[2]
            ref_cell   = cells[3]
            caho_cell  = cells[4]
        elif len(cells) == 3:
            date_cell  = cells[0]
            ref_cell   = cells[1]
            caho_cell  = cells[2]
        elif len(cells) == 4:
            # case_number still present but case_name rowspanned away
            current_case_number = cells[0].get_text(strip=True)
            date_cell  = cells[1]
            ref_cell   = cells[2]
            caho_cell  = cells[3]
        else:
            continue

        if not current_case_name or not current_case_name.strip():
            continue

        date_text = date_cell.get_text(strip=True)
        if not date_text or not date_text.replace('\xa0', '').strip():
            continue

        # Reference number and PDF URL
        ref_link = ref_cell.find('a')
        ref_raw  = (
            ref_link.get_text(strip=True) if ref_link
            else ref_cell.get_text(strip=True)
        )
        ref_text = re.sub(r'\s*\(PDF\)\s*', '', ref_raw).strip()

        raw_href = ref_link.get('href', '') if ref_link else ""
        pdf_url  = make_absolute_url(raw_href) if raw_href else None

        # Media ID from PDF URL
        media_id_m = (
            re.search(r'/media/(\d+)/', raw_href) or
            re.search(r'/d9/\d{4}-\d{2}/(.+?)\.pdf', raw_href) or
            re.search(r'/(\d{4,}[a-z]*)\.pdf', raw_href)
        )
        pdf_media_id = media_id_m.group(1) if media_id_m else None

        # CAHO decision flag (non-blank X in last column means CAHO decided it)
        caho_text = caho_cell.get_text(strip=True)
        is_caho_decision = bool(
            caho_text and caho_text.replace('\xa0', '').strip()
        )

        # Reference number parsing: "1671" → base=1671, sub=""
        #                            "1671a" → base=1671, sub="a"
        ref_match = re.match(r'^(\d+)([a-z]*)$', ref_text.strip().lower())
        ref_base       = int(ref_match.group(1)) if ref_match else None
        ref_sub_letter = ref_match.group(2) if ref_match else ""
        is_sub_decision = bool(ref_sub_letter)

        # Case type from case number (e.g., 2025C00038 or 19C00033)
        cn_clean = re.sub(r'\s+', '', current_case_number or "")
        case_type_m = re.match(r'^(\d{2,4})([ABC])(\d+)$', cn_clean)
        if case_type_m:
            raw_year         = case_type_m.group(1)
            case_year        = int(raw_year) + (2000 if len(raw_year) == 2 else 0)
            case_type_code   = case_type_m.group(2)
        else:
            case_year       = None
            case_type_code  = None
        case_type_desc = CASE_TYPE_MAP.get(case_type_code, "Unknown")

        # Unique key
        url = pdf_url or f"{BASE_URL}/eoir/ocaho-decision-{ref_text}"

        # Change-detection hash
        hash_source  = f"{current_case_name}{current_case_number}{date_text}{ref_text}"
        content_hash = hashlib.md5(hash_source.lower().encode('utf-8')).hexdigest()

        # Optional full PDF text
        full_text = ""
        pdf_download_attempted = False
        pdf_text_extracted = False
        text_word_count = 0

        if download_pdfs and pdf_url:
            pdf_download_attempted = True
            full_text, pdf_text_extracted = extract_pdf_text(pdf_url)
            text_word_count = len(full_text.split()) if full_text else 0

        decisions.append({
            "url":                    url,
            "content_hash":           content_hash,
            "case_name":              current_case_name,
            "case_number":            current_case_number,
            "decision_date":          date_text,
            "reference_number":       ref_text,
            "reference_base":         ref_base,
            "reference_sub_letter":   ref_sub_letter or None,
            "is_sub_decision":        is_sub_decision,
            "is_caho_decision":       is_caho_decision,
            "pdf_url":                pdf_url,
            "pdf_media_id":           str(pdf_media_id) if pdf_media_id else None,
            "full_text":              full_text,
            "text_word_count":        text_word_count,
            "case_type_code":         case_type_code,
            "case_type_description":  case_type_desc,
            "year_from_case_number":  case_year,
            "source_volume_name":     volume_name,
            "source_volume_url":      volume_url,
            "source_volume_number":   volume_num,
            "decision_type":          "OCAHO",
            "pdf_download_attempted": pdf_download_attempted,
            "pdf_text_extracted":     pdf_text_extracted,
            "scraped_date":           TODAY_STR,
        })

    return decisions


print("✅ Extraction functions loaded.")

In [ ]:
######################## SCRAPING LOGIC ##############################
#
# Follows the exact architecture from the Day 4 bill scraper:
#   Step 1 - Check for data history / load existing dataset
#   Step 2 - Set up change log + error log
#   Step 3 - Scrape volume listing pages, collect current decisions
#   Step 4 - Compare against history (hash check), detect changes
#   Step 5 - Write output: data JSON, changelog, error log
#
# Run this cell after Cell 1. Re-run daily (or via cron) for updates.
######################################################################

import os
import json
import hashlib
import datetime
import traceback
import requests
import time
import re
from bs4 import BeautifulSoup


######### CONFIGURATION #########

# Set to True to download each PDF and extract full text via pdftotext.
# WARNING: First run will download 1,700+ PDFs — very slow (~30-60 min).
# On subsequent runs only NEW decisions trigger downloads.
DOWNLOAD_PDFS = False

# BIA/AG volumes to scrape (newest first for changelog readability)
BIA_VOLUMES = [
    {"name": "Volume 29",  "url": "https://www.justice.gov/eoir/volume-29",                              "num": 29},
    {"name": "Volume 28",  "url": "https://www.justice.gov/eoir/volume-28",                              "num": 28},
    {"name": "Volume 27",  "url": "https://www.justice.gov/eoir/volume-27",                              "num": 27},
    {"name": "Volume 26",  "url": "https://www.justice.gov/eoir/vll/intdec/nfvol26.html",               "num": 26},
    {"name": "Volume 25",  "url": "https://www.justice.gov/eoir/vll/intdec/nfvol25.html",               "num": 25},
    {"name": "Volume 24",  "url": "https://www.justice.gov/eoir/vll/intdec/nfvol24.html",               "num": 24},
]

# OCAHO volumes to scrape (newest first)
OCAHO_VOLUMES = [
    {"name": "Looseleaf Volume 22", "url": "https://www.justice.gov/eoir/listing-volume-22-decisions", "num": 22},
    {"name": "Looseleaf Volume 21", "url": "https://www.justice.gov/eoir/listing-volume-21-decisions", "num": 21},
    {"name": "Looseleaf Volume 20", "url": "https://www.justice.gov/eoir/listing-volume-20-decisions", "num": 20},
    {"name": "Looseleaf Volume 19", "url": "https://www.justice.gov/eoir/listing-volume-19-decisions", "num": 19},
    {"name": "Looseleaf Volume 18", "url": "https://www.justice.gov/eoir/listing-volume-18-decisions", "num": 18},
    {"name": "Looseleaf Volume 17", "url": "https://www.justice.gov/eoir/listing-volume-17-decisions", "num": 17},
    {"name": "Looseleaf Volume 16", "url": "https://www.justice.gov/eoir/listing-volume-16-decisions", "num": 16},
    {"name": "Looseleaf Volume 15", "url": "https://www.justice.gov/eoir/listing-volume-15-decisions", "num": 15},
    {"name": "Looseleaf Volume 14", "url": "https://www.justice.gov/eoir/listing-volume-14-decisions", "num": 14},
]

# Output file paths
BIA_DATA_FILE   = "data/bia_ag_decisions.json"
OCAHO_DATA_FILE = "data/ocaho_decisions.json"
TODAY_STR       = datetime.date.today().isoformat()


#########STEP ONE#########
# Check for data history and load existing datasets

os.makedirs("data/changelogs",  exist_ok=True)
os.makedirs("data/error_logs",  exist_ok=True)
os.makedirs("data",             exist_ok=True)

if os.path.exists(BIA_DATA_FILE):
    print("Found existing BIA/AG dataset. Loading history...")
    with open(BIA_DATA_FILE, 'r') as f:
        old_bia_list = json.load(f)
    old_bia_map = {item['url']: item for item in old_bia_list}
    print(f"  Loaded {len(old_bia_map)} prior BIA/AG decisions.")
else:
    print("No existing BIA/AG dataset found. Initializing baseline run...")
    old_bia_map = {}  # empty map → everything treated as new

if os.path.exists(OCAHO_DATA_FILE):
    print("Found existing OCAHO dataset. Loading history...")
    with open(OCAHO_DATA_FILE, 'r') as f:
        old_ocaho_list = json.load(f)
    old_ocaho_map = {item['url']: item for item in old_ocaho_list}
    print(f"  Loaded {len(old_ocaho_map)} prior OCAHO decisions.")
else:
    print("No existing OCAHO dataset found. Initializing baseline run...")
    old_ocaho_map = {}


#########STEP TWO#########
# Set up change logs and error logs (one pair per dataset)

bia_changelog = {
    "date": TODAY_STR,
    "additions": [],
    "deletions": [],
    "modifications": []
}
bia_error_log = {"date": TODAY_STR, "errors": []}

ocaho_changelog = {
    "date": TODAY_STR,
    "additions": [],
    "deletions": [],
    "modifications": []
}
ocaho_error_log = {"date": TODAY_STR, "errors": []}


#########STEP THREE — BIA/AG#########
# Scrape each BIA/AG volume listing page

print("\n" + "="*60)
print("SCRAPING BIA/AG DECISIONS")
print("="*60)

todays_bia        = []
all_bia_urls_seen = set()

for vol in BIA_VOLUMES:
    vol_name = vol['name']
    vol_url  = vol['url']
    vol_num  = vol['num']
    print(f"\n  [{vol_name}] Fetching {vol_url}...")
    try:
        time.sleep(1)  # polite crawl delay
        response = requests.get(vol_url, headers=HEADERS, timeout=30, allow_redirects=True)
        response.raise_for_status()
        final_url = response.url
        print(f"    Landed at: {final_url}  ({len(response.content):,} bytes)")

        decisions = extract_bia_decisions(
            response.text, vol_name, final_url, vol_num,
            download_pdfs=DOWNLOAD_PDFS
        )
        print(f"    Extracted {len(decisions)} decisions.")

        for decision in decisions:
            url = decision['url']
            all_bia_urls_seen.add(url)

            if url not in old_bia_map:
                # Brand new decision
                if DOWNLOAD_PDFS and not decision.get('pdf_download_attempted') and decision.get('pdf_url'):
                    decision['pdf_download_attempted'] = True
                    decision['full_text'], decision['pdf_text_extracted'] = extract_pdf_text(decision['pdf_url'])
                    decision['text_word_count'] = len(decision['full_text'].split()) if decision['full_text'] else 0
                    time.sleep(1)
                todays_bia.append(decision)
                bia_changelog["additions"].append({
                    "decision_id":  decision.get("decision_id"),
                    "case_name":    decision.get("case_name"),
                    "citation":     decision.get("citation_full"),
                    "url":          url
                })
            else:
                old_item = old_bia_map[url]
                if old_item['content_hash'] == decision['content_hash']:
                    # No change detected — carry forward existing record
                    # (preserves any PDF text already extracted in prior runs)
                    todays_bia.append(old_item)
                else:
                    # Content changed — update record, preserve existing PDF text
                    if not decision.get('full_text') and old_item.get('full_text'):
                        decision['full_text']          = old_item['full_text']
                        decision['text_word_count']    = old_item.get('text_word_count', 0)
                        decision['pdf_text_extracted'] = old_item.get('pdf_text_extracted', False)
                    todays_bia.append(decision)
                    # Log which fields changed
                    meaningful_changes = {
                        k: {"from": old_item.get(k), "to": v}
                        for k, v in decision.items()
                        if k not in ('scraped_date', 'content_hash') and old_item.get(k) != v
                    }
                    if meaningful_changes:
                        bia_changelog["modifications"].append({
                            "decision_id": decision.get("decision_id"),
                            "case_name":   decision.get("case_name"),
                            "url":         url,
                            "changes":     meaningful_changes
                        })
                        print(f"    ~ MODIFIED: {decision.get('case_name')} ({decision.get('citation_full')})")

    except Exception as e:
        print(f"  ❌ Error scraping {vol_name}: {str(e)}")
        # Keep yesterday's decisions from this volume if available
        for url, old_item in old_bia_map.items():
            if old_item.get('source_volume_name') == vol_name and url not in all_bia_urls_seen:
                todays_bia.append(old_item)
                all_bia_urls_seen.add(url)
        bia_error_log["errors"].append({
            "volume":     vol_name,
            "url":        vol_url,
            "error_type": type(e).__name__,
            "message":    str(e),
            "traceback":  traceback.format_exc().splitlines()[-3:]
        })

# Check for deletions: decisions present yesterday but not in today's scrape
for url, old_item in old_bia_map.items():
    if url not in all_bia_urls_seen:
        bia_changelog["deletions"].append({
            "decision_id": old_item.get("decision_id"),
            "case_name":   old_item.get("case_name"),
            "citation":    old_item.get("citation_full"),
            "url":         url
        })

print(f"\n  BIA/AG total: {len(todays_bia)} decisions")
print(f"  Additions: {len(bia_changelog['additions'])}  "
      f"Modifications: {len(bia_changelog['modifications'])}  "
      f"Deletions: {len(bia_changelog['deletions'])}")


#########STEP THREE — OCAHO#########
# Scrape each OCAHO volume listing page

print("\n" + "="*60)
print("SCRAPING OCAHO DECISIONS")
print("="*60)

todays_ocaho        = []
all_ocaho_urls_seen = set()

for vol in OCAHO_VOLUMES:
    vol_name = vol['name']
    vol_url  = vol['url']
    vol_num  = vol['num']
    print(f"\n  [{vol_name}] Fetching {vol_url}...")
    try:
        time.sleep(1)
        response = requests.get(vol_url, headers=HEADERS, timeout=30, allow_redirects=True)
        response.raise_for_status()
        final_url = response.url
        print(f"    Landed at: {final_url}  ({len(response.content):,} bytes)")

        decisions = extract_ocaho_decisions(
            response.text, vol_name, final_url, vol_num,
            download_pdfs=DOWNLOAD_PDFS
        )
        print(f"    Extracted {len(decisions)} decisions.")

        for decision in decisions:
            url = decision['url']
            all_ocaho_urls_seen.add(url)

            if url not in old_ocaho_map:
                if DOWNLOAD_PDFS and not decision.get('pdf_download_attempted') and decision.get('pdf_url'):
                    decision['pdf_download_attempted'] = True
                    decision['full_text'], decision['pdf_text_extracted'] = extract_pdf_text(decision['pdf_url'])
                    decision['text_word_count'] = len(decision['full_text'].split()) if decision['full_text'] else 0
                    time.sleep(1)
                todays_ocaho.append(decision)
                ocaho_changelog["additions"].append({
                    "reference_number": decision.get("reference_number"),
                    "case_name":        decision.get("case_name"),
                    "decision_date":    decision.get("decision_date"),
                    "url":              url
                })
            else:
                old_item = old_ocaho_map[url]
                if old_item['content_hash'] == decision['content_hash']:
                    todays_ocaho.append(old_item)
                else:
                    if not decision.get('full_text') and old_item.get('full_text'):
                        decision['full_text']          = old_item['full_text']
                        decision['text_word_count']    = old_item.get('text_word_count', 0)
                        decision['pdf_text_extracted'] = old_item.get('pdf_text_extracted', False)
                    todays_ocaho.append(decision)
                    meaningful_changes = {
                        k: {"from": old_item.get(k), "to": v}
                        for k, v in decision.items()
                        if k not in ('scraped_date', 'content_hash') and old_item.get(k) != v
                    }
                    if meaningful_changes:
                        ocaho_changelog["modifications"].append({
                            "reference_number": decision.get("reference_number"),
                            "case_name":        decision.get("case_name"),
                            "url":              url,
                            "changes":          meaningful_changes
                        })
                        print(f"    ~ MODIFIED: {decision.get('case_name')} (Ref# {decision.get('reference_number')})")

    except Exception as e:
        print(f"  ❌ Error scraping {vol_name}: {str(e)}")
        for url, old_item in old_ocaho_map.items():
            if old_item.get('source_volume_name') == vol_name and url not in all_ocaho_urls_seen:
                todays_ocaho.append(old_item)
                all_ocaho_urls_seen.add(url)
        ocaho_error_log["errors"].append({
            "volume":     vol_name,
            "url":        vol_url,
            "error_type": type(e).__name__,
            "message":    str(e),
            "traceback":  traceback.format_exc().splitlines()[-3:]
        })

for url, old_item in old_ocaho_map.items():
    if url not in all_ocaho_urls_seen:
        ocaho_changelog["deletions"].append({
            "reference_number": old_item.get("reference_number"),
            "case_name":        old_item.get("case_name"),
            "url":              url
        })

print(f"\n  OCAHO total: {len(todays_ocaho)} decisions")
print(f"  Additions: {len(ocaho_changelog['additions'])}  "
      f"Modifications: {len(ocaho_changelog['modifications'])}  "
      f"Deletions: {len(ocaho_changelog['deletions'])}")

print(f"\n  ━━━ GRAND TOTAL: {len(todays_bia) + len(todays_ocaho)} decision rows ━━━")


#########STEP FOUR#########
# Write all output files

print("\n" + "="*60)
print("WRITING OUTPUT FILES")
print("="*60)

# BIA/AG data — sorted descending by volume then decision ID
with open(BIA_DATA_FILE, 'w') as f:
    sorted_bia = sorted(
        todays_bia,
        key=lambda x: (
            -(x.get('source_volume_number') or 0),
            -(x.get('decision_id') or 0)
        )
    )
    json.dump(sorted_bia, f, indent=2)
print(f"  ✓ {BIA_DATA_FILE}  ({len(sorted_bia)} records)")

# OCAHO data — sorted descending by volume then reference number
with open(OCAHO_DATA_FILE, 'w') as f:
    sorted_ocaho = sorted(
        todays_ocaho,
        key=lambda x: (
            -(x.get('source_volume_number') or 0),
            -(x.get('reference_base') or 0),
            x.get('reference_sub_letter') or ''
        )
    )
    json.dump(sorted_ocaho, f, indent=2)
print(f"  ✓ {OCAHO_DATA_FILE}  ({len(sorted_ocaho)} records)")

# BIA/AG changelog (only write if there are changes)
if any([bia_changelog["additions"], bia_changelog["deletions"], bia_changelog["modifications"]]):
    cl_path = f"data/changelogs/{TODAY_STR}_bia_ag.json"
    with open(cl_path, 'w') as f:
        json.dump(bia_changelog, f, indent=2)
    print(f"  ✓ {cl_path}  "
          f"(+{len(bia_changelog['additions'])} "
          f"~{len(bia_changelog['modifications'])} "
          f"-{len(bia_changelog['deletions'])})")

# OCAHO changelog
if any([ocaho_changelog["additions"], ocaho_changelog["deletions"], ocaho_changelog["modifications"]]):
    cl_path = f"data/changelogs/{TODAY_STR}_ocaho.json"
    with open(cl_path, 'w') as f:
        json.dump(ocaho_changelog, f, indent=2)
    print(f"  ✓ {cl_path}  "
          f"(+{len(ocaho_changelog['additions'])} "
          f"~{len(ocaho_changelog['modifications'])} "
          f"-{len(ocaho_changelog['deletions'])})")

# BIA/AG error log (only write if errors occurred)
if bia_error_log["errors"]:
    el_path = f"data/error_logs/{TODAY_STR}_bia_ag.json"
    with open(el_path, 'w') as f:
        json.dump(bia_error_log, f, indent=2)
    print(f"  ⚠️  {el_path}  ({len(bia_error_log['errors'])} errors)")

# OCAHO error log
if ocaho_error_log["errors"]:
    el_path = f"data/error_logs/{TODAY_STR}_ocaho.json"
    with open(el_path, 'w') as f:
        json.dump(ocaho_error_log, f, indent=2)
    print(f"  ⚠️  {el_path}  ({len(ocaho_error_log['errors'])} errors)")

print("\n✅ Scrape complete!")